In [217]:
# Imported Sionna and the functions we will use
# Imported the NumPy library for mathematical operations
import sionna.rt
import mitsuba as mi
from sionna.rt import (
    PlanarArray,
    Transmitter,
    Receiver,
    load_scene,
    PathSolver,
    SceneObject,
    ITURadioMaterial
    )
import numpy as np

In [218]:
# Called our environment and loaded the scene
scene = load_scene()

In [219]:
# Set our frequency to 3.5 GHz
scene.frequency = 7e9

In [220]:
# Using the PlanarArray function, we defined the number of Tx antenna elements, to be used as our base station, antenna properties and orientation

scene.tx_array= PlanarArray(
    num_rows=1, 
    num_cols=1,
    pattern="iso", 
    polarization="V",)

In [221]:
# We defined the properties of our receiving antenna
scene.rx_array= PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V",
)

In [222]:
# Named our transmitter Base Station 1 and defined its position
bs = Transmitter(
    name="BS1",
    position=[0, 0, 0]
)

In [223]:
# Added our base station to the scene
scene.add(bs)

In [224]:
# Set r = 20 meters as a constant value
r = 20

In [225]:
# Named our first receiver and set its position at a distance of r from the base station
rx1 = Receiver(
    name="RX1",
    position=[r, 0, 0]
)

In [226]:
# Named our second receiver and set its position at a distance of 2*r from the base station# Named our second receiver and set its position at a distance of 2*r from the base station
rx2 = Receiver(
    name="RX2",
    position=[2*r, 0, 0]
)

In [227]:
# Named our third receiver and set its position at a distance of 3*r from the base station
rx3 = Receiver(
    name="RX3",
    position=[3*r, 0, 0]
)

In [228]:
# Named our fourth receiver and set its position at a distance of 4*r from the base station
rx4 = Receiver(
    name="RX4",
    position=[10*r, 0, 0]
)

In [229]:
# Added our receivers to the scene
scene.add(rx1)
scene.add(rx2)
scene.add(rx3)
scene.add(rx4)

In [230]:
# Printed the position of each receiver to make sure their positions were correct
for name in ["RX1", "RX2", "RX3", "RX4"]:
    rx = scene.get(name)
    print(name, np.array(rx.position))

RX1 [[20.]
 [ 0.]
 [ 0.]]
RX2 [[40.]
 [ 0.]
 [ 0.]]
RX3 [[60.]
 [ 0.]
 [ 0.]]
RX4 [[200.]
 [  0.]
 [  0.]]


In [231]:
# Defined the constant speed of light, wavelength, and base station power
# Calculated the free space path loss values for each receiver
# path gain = - path loss
# Calculated the received signal strength (RSS) for each receiver
c= 299792458.0
wavelength = c/ scene.frequency
bs_power_dbm = 30
fspl1 = 20*np.log10(4*np.pi*rx1.position[0]/wavelength)
fspl2 = 20*np.log10(4*np.pi*rx2.position[0]/wavelength)
fspl3 = 20*np.log10(4*np.pi*rx3.position[0]/wavelength)
fspl4 = 20*np.log10(4*np.pi*rx4.position[0]/wavelength)

rss1 = bs_power_dbm - fspl1
rss2 = bs_power_dbm - fspl2
rss3 = bs_power_dbm - fspl3
rss4 = bs_power_dbm - fspl4

print(f"RSS at RX1: {rss1} dBm")
print(f"RSS at RX2: {rss2} dBm")
print(f"RSS at RX3: {rss3} dBm")
print(f"RSS at RX4: {rss4} dBm")

print(f"FSPL at RX1: {fspl1} dB")
print(f"FSPL at RX2: {fspl2} dB")
print(f"FSPL at RX3: {fspl3} dB")
print(f"FSPL at RX4: {fspl4} dB")

# Convert received power from dBm to nanowatts for comparing
p_rx1 = 10**((rss1 - 30) / 10) * 1e9
p_rx2 = 10**((rss2 - 30) / 10) * 1e9
p_rx3 = 10**((rss3 - 30) / 10) * 1e9
p_rx4 = 10**((rss4 - 30) / 10) * 1e9

print(f"RX1: {p_rx1.item():.2f} nW")
print(f"RX2: {p_rx2.item():.2f} nW")
print(f"RX3: {p_rx3.item():.2f} nW")
print(f"RX4: {p_rx4.item():.2f} nW")

print(f"RX2 / RX1: {((p_rx2 / p_rx1).item()):.2f}")

RSS at RX1: [-45.370346] dBm
RSS at RX2: [-51.390945] dBm
RSS at RX3: [-54.912766] dBm
RSS at RX4: [-65.37034] dBm
FSPL at RX1: [75.370346] dB
FSPL at RX2: [81.390945] dB
FSPL at RX3: [84.912766] dB
FSPL at RX4: [95.37034] dB
RX1: 29.04 nW
RX2: 7.26 nW
RX3: 3.23 nW
RX4: 0.29 nW
RX2 / RX1: 0.25


In [232]:
# Create a building geometry using Mitsuba
building_mesh = mi.load_dict({
    "type": "cube",
    "to_world": mi.ScalarTransform4f()
        .translate([30, 0, -4])
        .scale([5, 5, 7.5])
})

print(type(building_mesh))

<class 'mitsuba.Mesh'>


In [233]:
# Create a Sionna scene object from the Mitsuba mesh
building1 = SceneObject(
    mi_mesh=building_mesh,
    name="building1",
    radio_material=ITURadioMaterial(
        name="concrete",
        itu_type="concrete"
    )
)

In [242]:
# Add the building to the scene
scene.add(building1)

In [246]:
# Displayed the scene preview
scene.preview()

In [ ]:
solver = PathSolver()

paths_with = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction= False
)

In [237]:
# Remove the building
scene.remove("building1")

# Calculate propagation paths without the building
paths_without = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction=False
)

In [238]:
#obje kapatırsa  rss ve path gain değerleri değişimi
# Get channel coefficients

# Calculate path gain and RSS from channel coefficients
def calculate_rss(paths, tx_power_dbm=30):

    a_real, a_imag = paths.a
    a = np.array(a_real) + 1j * np.array(a_imag)

    # Combine propagation paths coherently
    h = np.sum(a, axis=-1)

    # Calculate path gain
    path_gain = np.abs(h)**2
    path_gain_db = 10 * np.log10(path_gain)

    # Calculate received signal strength
    rss_dbm = tx_power_dbm + path_gain_db

    return path_gain_db.reshape(-1), rss_dbm.reshape(-1)

In [239]:
# With building
gain_with, rss_with = calculate_rss(paths_with)

# Without building
gain_without, rss_without = calculate_rss(paths_without)

# Compare results
for i in range(4):
    print(f"RX{i+1}")
    print(f"RSS without building: {rss_without[i]:.2f} dBm")
    print(f"RSS with building:    {rss_with[i]:.2f} dBm")
    print(f"RSS difference:       {rss_with[i] - rss_without[i]:.2f} dB")
    print()

RX1
RSS without building: -45.37 dBm
RSS with building:    -48.02 dBm
RSS difference:       -2.65 dB

RX2
RSS without building: -51.39 dBm
RSS with building:    -84.53 dBm
RSS difference:       -33.14 dB

RX3
RSS without building: -54.91 dBm
RSS with building:    -86.09 dBm
RSS difference:       -31.18 dB

RX4
RSS without building: -65.37 dBm
RSS with building:    -98.51 dBm
RSS difference:       -33.14 dB



In [240]:
scene.add(building1)

paths_los = solver(
    scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    diffraction=False
)

gain_los, rss_los = calculate_rss(paths_los)

print(f"RX1 RSS (LoS only): {rss_los[0]:.2f} dBm")

RX1 RSS (LoS only): -45.37 dBm


C:\Users\Elif\AppData\Local\Temp\ipykernel_22556\4013894590.py:15: RuntimeWarning: divide by zero encountered in log10
  path_gain_db = 10 * np.log10(path_gain)


In [245]:
# Calculate paths with diffraction enabled
paths_diff = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    diffraction=False
)

# Calculate paths with diffraction disabled
paths_no_diff = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    diffraction=False
)

# Calculate RSS
gain_diff, rss_diff = calculate_rss(paths_diff)
gain_no_diff, rss_no_diff = calculate_rss(paths_no_diff)

# Compare
for i in range(4):
    print(f"RX{i+1}")
    print(f"With diffraction:    {rss_diff[i]:.6f} dBm")
    print(f"Without diffraction: {rss_no_diff[i]:.6f} dBm")
    print()

RX1
With diffraction:    -45.370346 dBm
Without diffraction: -45.370346 dBm

RX2
With diffraction:    -84.529831 dBm
Without diffraction: -84.529831 dBm

RX3
With diffraction:    -88.051651 dBm
Without diffraction: -88.051651 dBm

RX4
With diffraction:    -98.509232 dBm
Without diffraction: -98.509232 dBm



#research question 10: which kind of materials effect the propogation more?

In [282]:
# Remove the concrete building from the scene
scene.remove("building3")

In [286]:
# Select the material manually
selected_material = "concrete"  # concrete, glass, brick, wood, metal

# Remove the previous building if it exists
if scene.get("building3") is not None:
    scene.remove("building3")

In [ ]:
    # Add building to scene
    scene.add(building3)

In [ ]:
# Calculate propagation paths
paths_material = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction=False
)

# Calculate RSS and path gain
gain_material, rss_material = calculate_rss(paths_material)

# Print results
print(f"\nMaterial: {selected_material.upper()}")

for i in range(4):
    print(f"RX{i+1}")
    print(f"  RSS:       {rss_material[i]:.2f} dBm")
    print(f"  Path Gain: {gain_material[i]:.2f} dB")



Material: CONCRETE
RX1
  RSS:       -48.02 dBm
  Path Gain: -78.02 dB
RX2
  RSS:       -84.53 dBm
  Path Gain: -114.53 dB
RX3
  RSS:       -86.09 dBm
  Path Gain: -116.09 dB
RX4
  RSS:       -98.51 dBm
  Path Gain: -128.51 dB
